# 🧬 Smart-seq2 Notebook 1: Quality Control & `fastp` Trimming

This notebook walks through:
1. Inspecting raw Smart-seq2 paired-end FASTQ reads
2. Automated `FastQC` execution on raw libraries
3. High-performance adapter and quality trimming using `fastp` (Nextera adapters, poly-A/G, quality sliding window)
4. Before vs. After QC metric comparisons

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_config
from src.sc_qc import SmartSeq2QC
from src.sc_preprocess import SmartSeq2Preprocessor

config = load_config('../config/pipeline_config.yaml')
print(f"Project: {config['project']['name']}")

### 1. Run Raw FastQC

In [ ]:
qc_engine = SmartSeq2QC(config)
qc_engine.run_fastqc_stage('raw')

### 2. Execute `fastp` Paired-End Trimming

In [ ]:
preprocessor = SmartSeq2Preprocessor(config)
trim_results = preprocessor.run_all()
df_trim = pd.DataFrame(trim_results)
df_trim

### 3. Visual Comparison: Read Filtering & Q30 Improvement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df_trim, x='cell_id', y='clean_reads', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Clean Reads per Cell')
axes[0].tick_params(axis='x', rotation=45)

df_q30 = df_trim.melt(id_vars=['cell_id'], value_vars=['raw_q30_rate', 'clean_q30_rate'], var_name='Stage', value_name='Q30_Rate')
sns.barplot(data=df_q30, x='cell_id', y='Q30_Rate', hue='Stage', ax=axes[1], palette='Set2')
axes[1].set_title('Q30 Rate Before vs After fastp')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()